In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_finance"

def ingest_raw_csv(filename: str, table_name: str, infer_schema: bool = False):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", infer_schema)
        .csv(f"{VOLUME_PATH}/{filename}")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    )
    print(f"Loaded {df.count():,} rows into {CATALOG}.{SCHEMA}.{table_name}")

# Keep transactions raw (strings) so corrupt records are captured in Silver quarantine
ingest_raw_csv("general_ledger_transactions.csv", "bronze_transactions", infer_schema=False)
ingest_raw_csv("chart_of_accounts.csv", "bronze_accounts", infer_schema=True)
ingest_raw_csv("budget_targets.csv", "bronze_budgets", infer_schema=True)